In [1]:
# physical_graph_path = "scenarios/cavia/1_26_solution_v1/physical_graph.graphml"
# app_graph_path = "scenarios/cavia/1_26_solution_v1/ms/1MMM.graphml"
# pkl_path = "scenarios/cavia/1_26_solution_v1/var_coeff_values_1MMM_slss.pkl"
# scenario_name = 1_26_solution_v1
# app_name = "1MMM"

from adapters.cavia.cavia_scenario_loader import CaviaScenarioLoader
from adapters.cavia.find_valid_scenarios import get_scenario_paths
from edge_sim_py.component_manager import ComponentManager

scenario_name = "1_26_solution_v0"
app_name = "1SMM"
phys_path, app_path, pkl_path = get_scenario_paths(scenario_name, app_name, force_rescan=False)
print("Loading scenario:", scenario_name)
print("Loading app:", app_name + "\n")

topology, apps, users = CaviaScenarioLoader(
    physical_graph_path=phys_path,
    app_graph_path=app_path,
    pkl_path=pkl_path,
    seed=42,
    dist="exponential",
).build_scenario()

print("Topology:", topology)
print("Application:", apps)
print("User:", users)

for a in apps:
    print(f"App {a.id} with services: {a.services}")

Loading scenario: 1_26_solution_v0
Loading app: 1SMM

Topology: Topology_1
Application: [Application_1]
User: [User_1]
App 1 with services: [Service_0, Service_1, Service_2, Service_3, Service_4, Service_5]


In [2]:
from pathlib import Path

from edge_sim_py.components.data_packet import DataPacket
from edge_sim_py.simulator import Simulator

BASE_DIR = Path.cwd()
dataset = ComponentManager.export_scenario(save_to_file=True, file_name=scenario_name, output_dir=BASE_DIR / "datasets")

def my_algorithm(parameters):
    return

def static_dummy_mobility(user):
    user.coordinates_trace.append(user.coordinates)

# Instantiating the simulator
simulator = Simulator(
    dump_interval=1,
    tick_unit="seconds",
    tick_duration=1,
    stopping_criterion = lambda model: all(d._status == 'finished' for d in DataPacket.all()) or model.schedule.steps >= 500,
    resource_management_algorithm=my_algorithm,
    user_defined_functions=[static_dummy_mobility]
)

# Loading our dataset
simulator.initialize(input_file=f"datasets/{scenario_name}.json")

# Running the simulation
simulator.run_model()

In [3]:
import msgpack
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 1000)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 1000)

def highlight_rows(row):
    if row["Time Step"] % 2 == 0:
        return ["background-color: #222222; color: white"] * len(row)
    else:
        return ["background-color: #333333; color: white"] * len(row)

def load_simulation_logs(logs_path="logs"):
    datasets = {}
    directory = Path(logs_path)
    
    if not directory.exists():
        print(f"Errore: La cartella {logs_path} non esiste.")
        return datasets

    for file_path in directory.glob("*.msgpack"):
        with open(file_path, "rb") as f:
            data = msgpack.unpackb(f.read(), strict_map_key=False)
            datasets[file_path.stem] = pd.DataFrame(data)
            
    return datasets

datasets = load_simulation_logs()

In [4]:
print(datasets.keys())
#datasets["Application"].copy().style.apply(highlight_rows, axis=1)
#datasets["EdgeServer"].copy().style.apply(highlight_rows, axis=1)
#d = datasets["User"].copy().style.apply(highlight_rows, axis=1)
datasets["User"].head(4).style.apply(highlight_rows, axis=1)
#datasets["Service"].copy().style.apply(highlight_rows, axis=1)
#datasets["NetworkFlow"].copy().style.apply(highlight_rows, axis=1)

dict_keys(['Service', 'Application', 'EdgeServer', 'User', 'DataPacket', 'NetworkSwitch'])


,Object,Time Step,Instance ID,Coordinates,Base Station,Delays,Delay SLAs,Communication Paths,Making Requests,Access History
0,User_1,0,1,"[2, 0]","BaseStation_14 ([2, 0])",{'1': None},{'1': 63.486},{},{'1': {'1': True}},"{'1': [{'start': 1, 'end': 1, 'duration': 1, 'waiting_time': 0, 'access_time': 0, 'interval': 501, 'next_access': 503}]}"
1,User_1,1,1,"[2, 0]","BaseStation_14 ([2, 0])",{'1': 27},{'1': 63.486},"{'1': [[14], [14], [14, 0], [0, 16], [16, 17], [17, 16, 9, 3, 14]]}","{'1': {'1': True, '2': False}}","{'1': [{'start': 1, 'end': 1, 'duration': 1, 'waiting_time': 0, 'access_time': 1, 'interval': 501, 'next_access': 503}]}"
2,User_1,2,1,"[2, 0]","BaseStation_14 ([2, 0])",{'1': 27},{'1': 63.486},"{'1': [[14], [14], [14, 0], [0, 16], [16, 17], [17, 16, 9, 3, 14]]}","{'1': {'1': True, '2': False, '3': False}}","{'1': [{'start': 1, 'end': 1, 'duration': 1, 'waiting_time': 0, 'access_time': 1, 'interval': 501, 'next_access': 503}]}"
3,User_1,3,1,"[2, 0]","BaseStation_14 ([2, 0])",{'1': 27},{'1': 63.486},"{'1': [[14], [14], [14, 0], [0, 16], [16, 17], [17, 16, 9, 3, 14]]}","{'1': {'1': True, '2': False, '3': False, '4': False}}","{'1': [{'start': 1, 'end': 1, 'duration': 1, 'waiting_time': 0, 'access_time': 1, 'interval': 501, 'next_access': 503}]}"


In [5]:
from edge_sim_py.components.data_packet import DataPacket
print(len(DataPacket.all()))

1


In [6]:
#datasets["DataPacket"].copy().style.apply(highlight_rows, axis=1)
df_finished = datasets["DataPacket"][datasets["DataPacket"]["Status"] == "finished"].copy()
#df_finished.head(1).style.apply(highlight_rows, axis=1)
df_finished.columns


Index(['Object', 'Time Step', 'Id', 'User', 'Application', 'Size', 'Status', 'Queue Delay', 'Transmission Delay', 'Processing Delay', 'Propagation Delay', 'Total Delay', 'Total Path', 'Hops'], dtype='str')

In [7]:
#df_finished[["Time Step", "Total Delay"]].head(1).style.apply(highlight_rows, axis=1)
df_finished.drop(columns=["Hops"]).head(2).style.apply(highlight_rows, axis=1)

,Object,Time Step,Id,User,Application,Size,Status,Queue Delay,Transmission Delay,Processing Delay,Propagation Delay,Total Delay,Total Path
64,DataPacket_1,65,1,1,1,0.000000,finished,0,0,54,27,81,"[[14], [14], [14, 0], [0, 16], [16, 17], [17, 16, 9, 3, 14]]"


In [8]:
df_finished[["Time Step", "Application", "Hops"]].head(2).style.apply(highlight_rows, axis=1)

,Time Step,Application,Hops
64,65,1,"[{'hop_index': 0, 'link_index': 0, 'source': 14, 'target': 14, 'start_time': 1, 'end_time': 1, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 0, 'propagation_delay': 0, 'min_bandwidth': 0, 'max_bandwidth': 0, 'avg_bandwidth': 0, 'data_input': 1e-09, 'data_output': 1e-09}, {'hop_index': 1, 'link_index': 0, 'source': 14, 'target': 14, 'start_time': 1, 'end_time': 2, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 1, 'propagation_delay': 0, 'min_bandwidth': 0, 'max_bandwidth': 0, 'avg_bandwidth': 0, 'data_input': 1e-09, 'data_output': 1e-09}, {'hop_index': 2, 'link_index': 0, 'source': 14, 'target': 0, 'start_time': 4, 'end_time': 13, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 9, 'propagation_delay': 5, 'min_bandwidth': 50.0, 'max_bandwidth': 50.0, 'avg_bandwidth': 50.0, 'data_input': 1e-09, 'data_output': 1e-09}, {'hop_index': 3, 'link_index': 0, 'source': 0, 'target': 16, 'start_time': 14, 'end_time': 37, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 23, 'propagation_delay': 5, 'min_bandwidth': 96.0, 'max_bandwidth': 96.0, 'avg_bandwidth': 96.0, 'data_input': 1e-09, 'data_output': 1e-09}, {'hop_index': 4, 'link_index': 0, 'source': 16, 'target': 17, 'start_time': 38, 'end_time': 59, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 21, 'propagation_delay': 4, 'min_bandwidth': 65.0, 'max_bandwidth': 65.0, 'avg_bandwidth': 65.0, 'data_input': 1e-09, 'data_output': 1e-09}, {'hop_index': 5, 'link_index': 0, 'source': 17, 'target': 16, 'start_time': 60, 'end_time': 60, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 0, 'propagation_delay': 4, 'min_bandwidth': 65.0, 'max_bandwidth': 65.0, 'avg_bandwidth': 65.0, 'data_input': 1e-09, 'data_output': 1e-09}, {'hop_index': 5, 'link_index': 1, 'source': 16, 'target': 9, 'start_time': 61, 'end_time': 61, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 0, 'propagation_delay': 3, 'min_bandwidth': 63.0, 'max_bandwidth': 63.0, 'avg_bandwidth': 63.0, 'data_input': 1e-09, 'data_output': 1e-09}, {'hop_index': 5, 'link_index': 2, 'source': 9, 'target': 3, 'start_time': 62, 'end_time': 62, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 0, 'propagation_delay': 3, 'min_bandwidth': 43.0, 'max_bandwidth': 43.0, 'avg_bandwidth': 43.0, 'data_input': 1e-09, 'data_output': 1e-09}, {'hop_index': 5, 'link_index': 3, 'source': 3, 'target': 14, 'start_time': 63, 'end_time': 63, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 0, 'propagation_delay': 3, 'min_bandwidth': 41.0, 'max_bandwidth': 41.0, 'avg_bandwidth': 41.0, 'data_input': 1e-09, 'data_output': 1e-09}]"


In [9]:
# Estraiamo gli hops del primo pacchetto finito
hops_list = df_finished.iloc[0]["Hops"]
df_hops = pd.DataFrame(hops_list)

# Selezioniamo le colonne chiave per capire il percorso e i ritardi
cols = ["hop_index", "link_index", "source", "target", "propagation_delay", "processing_delay", "transmission_delay", "data_input", "data_output"]
df_hops[cols]

,hop_index,link_index,source,target,propagation_delay,processing_delay,transmission_delay,data_input,data_output
0,0,0,14,14,0,0,0,1.000000e-09,1.000000e-09
1,1,0,14,14,0,1,0,1.000000e-09,1.000000e-09
2,2,0,14,0,5,9,0,1.000000e-09,1.000000e-09
3,3,0,0,16,5,23,0,1.000000e-09,1.000000e-09
4,4,0,16,17,4,21,0,1.000000e-09,1.000000e-09
5,5,0,17,16,4,0,0,1.000000e-09,1.000000e-09
6,5,1,16,9,3,0,0,1.000000e-09,1.000000e-09
7,5,2,9,3,3,0,0,1.000000e-09,1.000000e-09
8,5,3,3,14,3,0,0,1.000000e-09,1.000000e-09


In [10]:
hops_list1 = df_finished.iloc[0]["Hops"]
df_hops1 = pd.DataFrame(hops_list1)
df_hops1[cols]

,hop_index,link_index,source,target,propagation_delay,processing_delay,transmission_delay,data_input,data_output
0,0,0,14,14,0,0,0,1.000000e-09,1.000000e-09
1,1,0,14,14,0,1,0,1.000000e-09,1.000000e-09
2,2,0,14,0,5,9,0,1.000000e-09,1.000000e-09
3,3,0,0,16,5,23,0,1.000000e-09,1.000000e-09
4,4,0,16,17,4,21,0,1.000000e-09,1.000000e-09
5,5,0,17,16,4,0,0,1.000000e-09,1.000000e-09
6,5,1,16,9,3,0,0,1.000000e-09,1.000000e-09
7,5,2,9,3,3,0,0,1.000000e-09,1.000000e-09
8,5,3,3,14,3,0,0,1.000000e-09,1.000000e-09


In [11]:
datasets["Service"].head(7).copy().style.apply(highlight_rows, axis=1)

,Object,Time Step,Instance ID,Available,Server,Being Provisioned,Processing Time,Processing Output,Last Migration
0,Service_5,0,5,True,14,False,0,0.000000,None
1,Service_4,0,4,True,17,False,21,0.000000,None
2,Service_3,0,3,True,16,False,23,0.000000,None
3,Service_2,0,2,True,0,False,9,0.000000,None
4,Service_1,0,1,True,14,False,1,0.000000,None
5,Service_0,0,0,True,14,False,0,0.000000,None
6,Service_5,1,5,True,14,False,0,0.000000,None
